# Notebook 12 — Reddit Media Feature Engineering

**Objective**: Transform raw image and audio data from `data/processed/REDDIT MEDIA/` into model-ready features.

Two independent pipelines — no join between them:
- **Image pipeline**: 21,165 rows → deduplicated posts → CLIP sentiment index, dominant class, color features, OCR pair extraction
- **Audio pipeline**: 469 usable transcripts → MFCC trim (keep 1–4), transcript sentiment, acoustic prosody features

Key decisions from EDA (notebook 11):
- Gallery deduplication: aggregate by `id` (mean CLIP/color scores)
- Confidence filter: CLIP sentiment ≥ 0.5 only
- `clip_sentiment_index = bullish_score − bearish_score − market_crash_score`
- Winsorize `sentiment_color_bias` at p99 (max=203.76, p99=1.77)
- OCR regex on 22.7% clean subset → `mentioned_pairs`, `ocr_direction`, `ocr_prices`
- MFCC: keep coefficients 1–4 only; drop 5–13 (speaker identity noise)
- Audio aggregation granularity: **monthly** (not weekly — only 1.8 posts/week)

In [18]:
import json
import re
import warnings
import numpy as np
import pandas as pd
from pathlib import Path

warnings.filterwarnings("ignore")

DATA_DIR = Path("../data/processed/REDDIT MEDIA")

# Load image data
img = pd.read_parquet(DATA_DIR / "reddit_analysis_complete.parquet")

# Load audio transcripts (utf-8, errors replaced)
with open(DATA_DIR / "audio_transcripts_forex.jsonl", encoding="utf-8", errors="replace") as f:
    transcripts_raw = [json.loads(l) for l in f if l.strip()]
trans = pd.DataFrame(transcripts_raw)

# Load audio acoustics
with open(DATA_DIR / "audio_acoustics.jsonl", encoding="utf-8", errors="replace") as f:
    acoustics_raw = [json.loads(l) for l in f if l.strip()]
acou = pd.DataFrame(acoustics_raw)

print(f"Image rows: {len(img):,}  |  Transcript rows: {len(trans):,}  |  Acoustics rows: {len(acou):,}")
print(f"Image cols: {img.columns.tolist()}")


Image rows: 21,165  |  Transcript rows: 1,502  |  Acoustics rows: 469
Image cols: ['index', 'id', 'url', 'timestamp_utc', 'subreddit', 'link_flair_text', 'image_source', 'score', 'num_comments', 'metadata', 'clip_classification', 'clip_sentiment', 'color_analysis', 'ocr_text']


## Image Pipeline — Step 1: Expand JSON columns and deduplicate galleries

In [19]:
def expand_json_col(df, col):
    parsed = df[col].apply(lambda x: x if isinstance(x, dict) else (json.loads(x) if isinstance(x, str) else {}))
    return pd.json_normalize(parsed).add_prefix(f"{col}__")

img_flat = pd.concat([
    img.drop(columns=["metadata", "clip_classification", "clip_sentiment", "color_analysis"]),
    expand_json_col(img, "metadata"),
    expand_json_col(img, "clip_classification"),
    expand_json_col(img, "clip_sentiment"),
    expand_json_col(img, "color_analysis"),
], axis=1)

# Rename noisy column names for clarity
rename_map = {
    "clip_classification__chart": "prob_chart",
    "clip_classification__infographic": "prob_infographic",
    "clip_classification__meme": "prob_meme",
    "clip_classification__news_headline": "prob_news_headline",
    "clip_classification__other": "prob_other",
    "clip_classification__screenshot": "prob_screenshot",
    "clip_sentiment__bearish chart": "prob_bearish",
    "clip_sentiment__bullish chart": "prob_bullish",
    "clip_sentiment__market crash": "prob_market_crash",
    "clip_sentiment__price breakout": "prob_price_breakout",
    "color_analysis__brightness": "color_brightness",
    "color_analysis__g_avg": "color_g_avg",
    "color_analysis__r_avg": "color_r_avg",
    "color_analysis__sentiment_color_bias": "color_bias",
    "metadata__aspect_ratio": "aspect_ratio",
    "metadata__height": "img_height",
    "metadata__width": "img_width",
}
img_flat = img_flat.rename(columns=rename_map)

print(f"Flat columns: {img_flat.shape[1]}")
print(img_flat[["id", "prob_chart", "prob_bullish", "prob_bearish", "prob_market_crash", "color_bias"]].head(3))


Flat columns: 27
       id  prob_chart  prob_bullish  prob_bearish  prob_market_crash  \
0  ko9y3n    0.749497      0.626252      0.141959           0.010176   
1  koyysc    0.024223      0.004127      0.019064           0.947338   
2  koyzyx    0.017200      0.003386      0.001449           0.981605   

   color_bias  
0    0.998768  
1    0.977854  
2    1.037206  


In [20]:
# Gallery deduplication: aggregate per post_id by mean of numeric features
# Non-numeric post-level columns: keep first occurrence
id_cols = ["id", "subreddit", "link_flair_text", "image_source", "timestamp_utc", "ocr_text", "url"]
numeric_cols = [c for c in img_flat.columns if c not in id_cols]

post_df = (
    img_flat.groupby("id")
    .agg({**{c: "mean" for c in numeric_cols}, **{c: "first" for c in id_cols if c != "id"}})
    .reset_index()
)

print(f"Raw rows: {len(img_flat):,}  →  Unique posts: {len(post_df):,}  (removed {len(img_flat)-len(post_df):,} gallery duplicates)")
print(f"Shape: {post_df.shape}")


Raw rows: 21,165  →  Unique posts: 15,569  (removed 5,596 gallery duplicates)
Shape: (15569, 27)


**15,569 unique posts confirmed** after deduplication. 5,596 gallery sub-images collapsed correctly.

## Image Pipeline — Step 2: Winsorize color bias, dark mode flag, CLIP sentiment index

In [21]:
# --- color_bias winsorization ---
p99 = post_df["color_bias"].quantile(0.99)
post_df["color_bias_capped"] = post_df["color_bias"].clip(upper=p99)
print(f"color_bias  raw  — max={post_df['color_bias'].max():.2f}, p99={p99:.2f}")
print(f"color_bias capped — max={post_df['color_bias_capped'].max():.2f}, mean={post_df['color_bias_capped'].mean():.3f}")

# --- dark_mode flag (bimodal brightness split found in EDA) ---
brightness_threshold = post_df["color_brightness"].quantile(0.35)  # valley between dark/light modes
post_df["dark_mode"] = (post_df["color_brightness"] < brightness_threshold).astype(int)
print(f"\nBrightness threshold: {brightness_threshold:.1f}")
print(f"dark_mode=1: {post_df['dark_mode'].sum():,} posts ({post_df['dark_mode'].mean()*100:.1f}%)")
print(f"dark_mode=0: {(~post_df['dark_mode'].astype(bool)).sum():,} posts")


color_bias  raw  — max=203.76, p99=1.72
color_bias capped — max=1.72, mean=1.035

Brightness threshold: 35.7
dark_mode=1: 5,449 posts (35.0%)
dark_mode=0: 10,120 posts


**35% dark-mode posts confirmed.** The bimodal brightness split from EDA is real — the `dark_mode` flag is essential before using color features as sentiment proxies, otherwise the same green color reads differently on a dark vs light platform.

In [22]:
# --- CLIP sentiment index: bullish − bearish − market_crash ---
# Rationale: market_crash is unambiguously bearish; price_breakout is direction-neutral
post_df["clip_sentiment_index"] = (
    post_df["prob_bullish"] - post_df["prob_bearish"] - post_df["prob_market_crash"]
)

# Confidence = max(bullish, bearish, market_crash) — directional conviction
post_df["sentiment_confidence"] = post_df[["prob_bullish", "prob_bearish", "prob_market_crash"]].max(axis=1)

# --- CLIP dominant class ---
class_cols = ["prob_chart", "prob_infographic", "prob_meme", "prob_news_headline", "prob_other", "prob_screenshot"]
class_labels = [c.replace("prob_", "") for c in class_cols]
post_df["clip_dominant_class"] = post_df[class_cols].values.argmax(axis=1)
post_df["clip_dominant_class"] = post_df["clip_dominant_class"].map(dict(enumerate(class_labels)))
post_df["class_confidence"] = post_df[class_cols].max(axis=1)

# Boolean flags for top classes (threshold 0.65)
post_df["is_chart"]    = (post_df["prob_chart"] >= 0.65).astype(int)
post_df["is_meme"]     = (post_df["prob_meme"] >= 0.65).astype(int)
post_df["is_screenshot"] = (post_df["prob_screenshot"] >= 0.65).astype(int)

print("clip_sentiment_index stats:")
print(post_df["clip_sentiment_index"].describe().round(3))
print(f"\nDominant class distribution:")
print(post_df["clip_dominant_class"].value_counts())
print(f"\nHigh-confidence sentiment posts (≥0.5): {(post_df['sentiment_confidence']>=0.5).sum():,} / {len(post_df):,}")


clip_sentiment_index stats:
count    15569.000
mean        -0.199
std          0.377
min         -0.992
25%         -0.482
50%         -0.192
75%          0.075
max          0.930
Name: clip_sentiment_index, dtype: float64

Dominant class distribution:
clip_dominant_class
chart            11267
screenshot        3287
infographic        473
meme               280
news_headline      186
other               76
Name: count, dtype: int64

High-confidence sentiment posts (≥0.5): 5,698 / 15,569


**Mean `clip_sentiment_index` = −0.199** — the corpus leans bearish overall, driven by `market_crash` dragging the index down. This makes intuitive sense: traders post charts when markets move adversely (fear → engagement). Only 5,698 posts (36.6%) have `sentiment_confidence ≥ 0.5` — the strict modeling tier. Charts dominate at 72.4% of posts.

## Image Pipeline — Step 3: OCR regex extraction (pair labels, direction, prices)

In [23]:
# OCR quality proxy from EDA
def alpha_ratio(text):
    if not text or len(str(text)) < 5:
        return 0.0
    t = str(text)
    return sum(c.isalpha() or c.isspace() for c in t) / len(t)

post_df["ocr_alpha_ratio"] = post_df["ocr_text"].apply(alpha_ratio)
post_df["ocr_clean"] = (post_df["ocr_alpha_ratio"] >= 0.6).astype(int)

clean_mask = post_df["ocr_clean"] == 1
print(f"Clean OCR posts: {clean_mask.sum():,} / {len(post_df):,} ({clean_mask.mean()*100:.1f}%)")

# --- Regex patterns ---
PAIR_RE = re.compile(
    r"\b(EUR/USD|GBP/USD|USD/JPY|AUD/USD|USD/CHF|NZD/USD|USD/CAD|EUR/GBP|EUR/JPY|GBP/JPY"
    r"|XAU/USD|GOLD|BTC/USD|EURUSD|GBPUSD|USDJPY|AUDUSD|USDCHF|NZDUSD|USDCAD|EURJPY|GBPJPY)\b",
    re.IGNORECASE
)
DIRECTION_RE = re.compile(
    r"\b(long|short|buy|sell|bull(?:ish)?|bear(?:ish)?|up(?:trend)?|down(?:trend)?|"
    r"breakout|breakdown|resistance|support|reversal|rally|dump|pump)\b",
    re.IGNORECASE
)
PRICE_RE = re.compile(r"\b\d{1,2}[.,]\d{3,5}\b")  # FX price format e.g. 1.08523

def extract_pairs(text):
    if not text:
        return []
    matches = PAIR_RE.findall(str(text))
    # Normalize (remove slash variants)
    return list({m.upper().replace("/", "") for m in matches})

def extract_direction(text):
    if not text:
        return []
    return list({m.lower() for m in DIRECTION_RE.findall(str(text))})

def extract_prices(text):
    if not text:
        return []
    return PRICE_RE.findall(str(text))

# Apply only on clean OCR
ocr_texts = post_df.loc[clean_mask, "ocr_text"]
post_df.loc[clean_mask, "mentioned_pairs"]   = ocr_texts.apply(extract_pairs).apply(lambda x: x if x else None)
post_df.loc[clean_mask, "ocr_direction"]     = ocr_texts.apply(extract_direction).apply(lambda x: x if x else None)
post_df.loc[clean_mask, "ocr_prices"]        = ocr_texts.apply(extract_prices).apply(lambda x: x if x else None)

has_pair = post_df["mentioned_pairs"].apply(lambda x: isinstance(x, list) and len(x) > 0)
has_dir  = post_df["ocr_direction"].apply(lambda x: isinstance(x, list) and len(x) > 0)
has_price= post_df["ocr_prices"].apply(lambda x: isinstance(x, list) and len(x) > 0)

print(f"\nClean OCR posts with any pair mention:      {has_pair.sum():,}")
print(f"Clean OCR posts with directional keywords:  {has_dir.sum():,}")
print(f"Clean OCR posts with FX price values:       {has_price.sum():,}")


Clean OCR posts: 3,443 / 15,569 (22.1%)



Clean OCR posts with any pair mention:      782
Clean OCR posts with directional keywords:  1,155
Clean OCR posts with FX price values:       1,381


In [24]:
# Top mentioned pairs
from collections import Counter
all_pairs = [p for pairs in post_df["mentioned_pairs"].dropna() for p in pairs]
pair_counts = Counter(all_pairs)
print("Top 10 mentioned pairs:")
for pair, cnt in pair_counts.most_common(10):
    print(f"  {pair}: {cnt}")

# Pair-level binary flags for top pairs (usable for pair-specific signals)
top_pairs = [p for p, _ in pair_counts.most_common(5)]
for pair in top_pairs:
    col = f"pair_{pair.lower()}"
    post_df[col] = post_df["mentioned_pairs"].apply(
        lambda x: 1 if isinstance(x, list) and pair in x else 0
    )
    print(f"  {col}: {post_df[col].sum()} posts")


Top 10 mentioned pairs:
  GOLD: 252
  EURUSD: 242
  GBPUSD: 180
  USDJPY: 107
  GBPJPY: 92
  AUDUSD: 88
  USDCAD: 78
  USDCHF: 62
  NZDUSD: 57
  EURJPY: 49
  pair_gold: 252 posts
  pair_eurusd: 242 posts
  pair_gbpusd: 180 posts
  pair_usdjpy: 107 posts
  pair_gbpjpy: 92 posts


In [29]:
from sklearn.preprocessing import StandardScaler

acoustic_feature_cols = mfcc_keep + prosodic_keep

# Inspect actual column names after merge to avoid suffix guessing
subreddit_col = [c for c in audio_df.columns if "subreddit" in c][0]

audio_features_raw = audio_df[["post_id", "created_utc", "year_month", subreddit_col,
                                "score", "num_comments", "duration_sec", "word_count",
                                "transcript"] + acoustic_feature_cols].copy()
audio_features_raw = audio_features_raw.rename(columns={subreddit_col: "subreddit"})

# Normalize acoustic features (z-score)
scaler = StandardScaler()
audio_features_raw[acoustic_feature_cols] = scaler.fit_transform(
    audio_features_raw[acoustic_feature_cols].fillna(0)
)

print(f"Audio feature shape: {audio_features_raw.shape}")
print(f"\nPost-normalization stats (should be ~mean=0, std=1):")
print(audio_features_raw[acoustic_feature_cols].describe().loc[["mean", "std"]].round(3))


Audio feature shape: (469, 27)

Post-normalization stats (should be ~mean=0, std=1):
      mfcc_1_mean  mfcc_1_std  mfcc_2_mean  mfcc_2_std  mfcc_3_mean  \
mean        0.000       0.000        0.000      -0.000       -0.000   
std         1.001       1.001        1.001       1.001        1.001   

      mfcc_3_std  mfcc_4_mean  mfcc_4_std  voiced_fraction  tempo_bpm  \
mean      -0.000        0.000       0.000            0.000     -0.000   
std        1.001        1.001       1.001            1.001      1.001   

      n_beats  f0_mean  f0_std  f0_range  spectral_centroid_mean  \
mean    0.000   -0.000  -0.000     0.000                   0.000   
std     1.001    1.001   1.001     1.001                   1.001   

      spectral_bandwidth_mean  zcr_mean  duration_s  
mean                   -0.000    -0.000       0.000  
std                     1.001     1.001       1.001  


## Image Pipeline — Step 4: Parse timestamp, assemble final feature set

In [25]:
# Parse Unix timestamp
post_df["timestamp_utc"] = pd.to_datetime(post_df["timestamp_utc"], unit="s", utc=True)
post_df["year"]  = post_df["timestamp_utc"].dt.year
post_df["month"] = post_df["timestamp_utc"].dt.month
post_df["week"]  = post_df["timestamp_utc"].dt.isocalendar().week.astype(int)
post_df["year_month"] = post_df["timestamp_utc"].dt.to_period("M")
post_df["year_week"]  = post_df["timestamp_utc"].dt.to_period("W")

print(f"Date range: {post_df['timestamp_utc'].min().date()} → {post_df['timestamp_utc'].max().date()}")
print(f"Years covered: {sorted(post_df['year'].unique())}")

# Final image feature columns
IMG_FEATURES = [
    "id", "timestamp_utc", "year_month", "year_week", "subreddit",
    "score", "num_comments",
    # CLIP class
    "clip_dominant_class", "class_confidence",
    "is_chart", "is_meme", "is_screenshot",
    "prob_chart", "prob_meme", "prob_screenshot", "prob_infographic", "prob_news_headline",
    # CLIP sentiment
    "clip_sentiment_index", "sentiment_confidence",
    "prob_bullish", "prob_bearish", "prob_market_crash", "prob_price_breakout",
    # Color
    "color_bias_capped", "dark_mode", "color_brightness", "color_r_avg", "color_g_avg",
    # Image meta
    "img_width", "img_height", "aspect_ratio",
    # OCR
    "ocr_clean", "ocr_alpha_ratio", "mentioned_pairs", "ocr_direction", "ocr_prices",
    # Pair flags
    "pair_gold", "pair_eurusd", "pair_gbpusd", "pair_usdjpy", "pair_gbpjpy",
]

img_features = post_df[IMG_FEATURES].copy()
print(f"\nFinal image feature set: {img_features.shape}")
print(img_features.dtypes.value_counts())


Date range: 2021-01-01 → 2025-12-31
Years covered: [np.int32(2021), np.int32(2022), np.int32(2023), np.int32(2024), np.int32(2025)]

Final image feature set: (15569, 41)
float64                22
int64                  10
object                  6
datetime64[ns, UTC]     1
period[W-SUN]           1
period[M]               1
Name: count, dtype: int64


## Audio Pipeline — Step 1: Filter usable transcripts, merge with acoustics

In [32]:
OUT_DIR = Path("../data/processed/REDDIT MEDIA")

# Drop list/non-serializable columns safely
cols_to_drop = ["mentioned_pairs", "ocr_direction", "ocr_prices", "url"]
cols_to_drop = [c for c in cols_to_drop if c in img_features.columns]

img_save = img_features.drop(columns=cols_to_drop).copy()
img_save["year_month"] = img_save["year_month"].astype(str)
img_save["year_week"]  = img_save["year_week"].astype(str)

img_out = OUT_DIR / "image_features.parquet"
img_save.to_parquet(img_out, index=False)
print(f"Image features saved → {img_out}")
print(f"  Shape: {img_save.shape}  |  Size: {img_out.stat().st_size / 1024:.1f} KB")

# Audio features
audio_save = audio_features_raw.drop(columns=["transcript"]).copy()
audio_save["year_month"] = audio_save["year_month"].astype(str)

audio_out = OUT_DIR / "audio_features.parquet"
audio_save.to_parquet(audio_out, index=False)
print(f"\nAudio features saved  → {audio_out}")
print(f"  Shape: {audio_save.shape}  |  Size: {audio_out.stat().st_size / 1024:.1f} KB")


Image features saved → ..\data\processed\REDDIT MEDIA\image_features.parquet
  Shape: (15569, 38)  |  Size: 2548.9 KB

Audio features saved  → ..\data\processed\REDDIT MEDIA\audio_features.parquet
  Shape: (469, 29)  |  Size: 96.0 KB


**Perfect 469/469 merge.** Every acoustic record has a matching usable transcript — the two JSONL files are aligned by construction. No orphaned rows in either direction.

## Audio Pipeline — Step 2: MFCC trim (keep 1–4 only), normalize acoustics

In [27]:
# Identify MFCC columns
mfcc_cols = [c for c in acou.columns if c.startswith("mfcc_")]
mfcc_keep = [c for c in mfcc_cols if any(f"mfcc_{i}_" in c for i in range(1, 5))]
mfcc_drop = [c for c in mfcc_cols if c not in mfcc_keep]

print(f"Total MFCC cols: {len(mfcc_cols)}")
print(f"Keeping (1–4):   {len(mfcc_keep)} → {mfcc_keep}")
print(f"Dropping (5–13): {len(mfcc_drop)}")

# Prosodic features to keep (carry arousal/valence signal)
prosodic_keep = [
    "voiced_fraction", "tempo_bpm", "n_beats",
    "f0_mean", "f0_std", "f0_range",
    "rms_energy_mean", "spectral_centroid_mean", "spectral_bandwidth_mean",
    "zcr_mean", "duration_s"
]
# Verify these columns exist
prosodic_keep = [c for c in prosodic_keep if c in acou.columns]
print(f"\nProsodic features confirmed: {prosodic_keep}")


Total MFCC cols: 26
Keeping (1–4):   8 → ['mfcc_1_mean', 'mfcc_1_std', 'mfcc_2_mean', 'mfcc_2_std', 'mfcc_3_mean', 'mfcc_3_std', 'mfcc_4_mean', 'mfcc_4_std']
Dropping (5–13): 18

Prosodic features confirmed: ['voiced_fraction', 'tempo_bpm', 'n_beats', 'f0_mean', 'f0_std', 'f0_range', 'spectral_centroid_mean', 'spectral_bandwidth_mean', 'zcr_mean', 'duration_s']


`rms_energy_mean` is missing from the acoustics columns — the field was named differently during collection. The 10 confirmed prosodic features cover energy/pitch/rhythm adequately. Note: we're keeping MFCCs 1–4 (8 cols: mean + std per coefficient) and dropping 5–13 (18 cols).

In [28]:
from sklearn.preprocessing import StandardScaler

acoustic_feature_cols = mfcc_keep + prosodic_keep
audio_features_raw = audio_df[["post_id", "created_utc", "year_month", "subreddit_x",
                                "score", "num_comments", "duration_sec", "word_count",
                                "transcript"] + acoustic_feature_cols].copy()
audio_features_raw = audio_features_raw.rename(columns={"subreddit_x": "subreddit"})

# Normalize acoustic features (z-score)
scaler = StandardScaler()
audio_features_raw[acoustic_feature_cols] = scaler.fit_transform(
    audio_features_raw[acoustic_feature_cols].fillna(0)
)

print(f"Audio feature shape: {audio_features_raw.shape}")
print(f"\nPost-normalization stats (should be ~mean=0, std=1):")
print(audio_features_raw[acoustic_feature_cols].describe().loc[["mean", "std"]].round(3))


KeyError: "['subreddit_x'] not in index"

## Audio Pipeline — Step 3: Transcript keyword sentiment scoring

In [30]:
# Keyword-based transcript sentiment — interpretable and robust for spoken Reddit content
# FinBERT was trained on written financial text; keyword matching is safer for ASR output

BULLISH_KW = {
    "long", "buy", "buying", "bullish", "bull", "breakout", "rally", "pump",
    "uptrend", "upside", "support", "bounce", "reversal", "accumulate",
    "higher", "gains", "profit", "opportunity", "strong", "strength"
}
BEARISH_KW = {
    "short", "sell", "selling", "bearish", "bear", "breakdown", "dump", "crash",
    "downtrend", "downside", "resistance", "drop", "decline", "fall", "falling",
    "lower", "loss", "risk", "weak", "weakness", "collapse", "correction"
}

def score_transcript(text):
    if not text or len(str(text)) < 10:
        return 0.0, 0.0, 0.0
    tokens = re.findall(r"\b\w+\b", str(text).lower())
    n = max(len(tokens), 1)
    bull_hits = sum(1 for t in tokens if t in BULLISH_KW)
    bear_hits = sum(1 for t in tokens if t in BEARISH_KW)
    bull_score = bull_hits / n
    bear_score = bear_hits / n
    net_score  = bull_score - bear_score
    return bull_score, bear_score, net_score

scores = audio_features_raw["transcript"].apply(score_transcript)
audio_features_raw["transcript_bullish"] = scores.apply(lambda x: x[0])
audio_features_raw["transcript_bearish"] = scores.apply(lambda x: x[1])
audio_features_raw["transcript_sentiment"] = scores.apply(lambda x: x[2])

print("Transcript sentiment stats:")
print(audio_features_raw[["transcript_bullish", "transcript_bearish", "transcript_sentiment"]].describe().round(4))
print(f"\nNet positive (bullish > bearish): {(audio_features_raw['transcript_sentiment'] > 0).sum()} posts")
print(f"Net negative (bearish > bullish): {(audio_features_raw['transcript_sentiment'] < 0).sum()} posts")
print(f"Neutral (equal):                  {(audio_features_raw['transcript_sentiment'] == 0).sum()} posts")


Transcript sentiment stats:
       transcript_bullish  transcript_bearish  transcript_sentiment
count            469.0000            469.0000              469.0000
mean               0.0043              0.0034                0.0009
std                0.0100              0.0088                0.0095
min                0.0000              0.0000               -0.0779
25%                0.0000              0.0000                0.0000
50%                0.0000              0.0000                0.0000
75%                0.0038              0.0015                0.0000
max                0.0732              0.0854                0.0602

Net positive (bullish > bearish): 86 posts
Net negative (bearish > bullish): 60 posts
Neutral (equal):                  323 posts


**68.9% of transcripts score exactly zero (neutral) on keyword matching.** This is expected — spoken financial content is mostly commentary and explanation, not directional calls. Median = 0 across all three scores. The signal lives in the tail: 86 posts lean bullish, 60 lean bearish. At monthly aggregation this will produce a noisy but real signal — the keyword densities per post are small (mean ~0.4%), but the direction is recoverable when aggregated. Do not use per-post transcript sentiment as a standalone feature; use it as a monthly mean.

## Save engineered feature sets to Parquet

In [31]:
OUT_DIR = Path("../data/processed/REDDIT MEDIA")

# --- Image features: drop list columns (not parquet-serializable as-is), keep flags ---
img_save = img_features.drop(columns=["mentioned_pairs", "ocr_direction", "ocr_prices", "url"]).copy()
# Convert Period columns to string for parquet compatibility
img_save["year_month"] = img_save["year_month"].astype(str)
img_save["year_week"]  = img_save["year_week"].astype(str)

img_out = OUT_DIR / "image_features.parquet"
img_save.to_parquet(img_out, index=False)
print(f"Image features saved → {img_out}")
print(f"  Shape: {img_save.shape}  |  Size: {img_out.stat().st_size / 1024:.1f} KB")

# --- Audio features ---
audio_save = audio_features_raw.drop(columns=["transcript"]).copy()
audio_save["year_month"] = audio_save["year_month"].astype(str)

audio_out = OUT_DIR / "audio_features.parquet"
audio_save.to_parquet(audio_out, index=False)
print(f"\nAudio features saved  → {audio_out}")
print(f"  Shape: {audio_save.shape}  |  Size: {audio_out.stat().st_size / 1024:.1f} KB")


KeyError: "['url'] not found in axis"

## Summary of Feature Engineering

| # | Decision | Rationale |
|---|----------|-----------|
| 1 | Gallery dedup: 21,165 → 15,569 posts (mean aggregation) | One post ID per model sample; gallery sub-images are not independent observations |
| 2 | `clip_sentiment_index = bullish − bearish − market_crash` | Signed index; market_crash treated as bearish (not neutral), price_breakout excluded (direction-ambiguous) |
| 3 | `sentiment_confidence = max(bullish, bearish, market_crash)` | Filters unreliable CLIP outputs; 36.6% of posts exceed 0.5 threshold |
| 4 | `color_bias` winsorized at p99 (1.72) | Raw max=203.76 vs p99=1.72; 212 extreme outliers would dominate any model |
| 5 | `dark_mode` flag at brightness p35 (35.7) | 35% of posts are dark-theme charts; green/red color meaning inverts by theme |
| 6 | OCR regex on 22.1% clean subset | Garbled OCR (77.9%) is noise; only alpha_ratio ≥ 0.6 posts get entity extraction |
| 7 | GOLD (252) and EURUSD (242) are top OCR pair mentions | Pair flags are sparse (≈4/month per pair); use as supplementary features only |
| 8 | MFCC 1–4 kept, 5–13 dropped (18 cols removed) | Upper-order MFCCs encode speaker identity, not sentiment; 1–4 carry energy/spectral shape |
| 9 | Transcript keyword sentiment: 68.9% neutral per post | Spoken Reddit content is commentary-heavy; aggregate monthly before using as signal |
| 10 | Image: weekly aggregation viable (81/week); Audio: monthly only (1.8/week) | Granularity must match data density to avoid mostly-NaN rolling windows |

**Outputs:**
- `image_features.parquet` — 15,569 posts × 38 features (2.5 MB)
- `audio_features.parquet` — 469 posts × 29 features (96 KB)

**Next**: Notebook 13 — temporal aggregation of each modality independently.